# GTSRB Robust Traffic Sign Recognition Workflow

Run this notebook in Colab with a GPU. It installs the project, creates EDA figures, trains the baseline and robust model, evaluates clean/corrupted performance, and writes report-ready artifacts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/gtsrb-robustness'
DRIVE_OUT = '/content/drive/MyDrive/gtsrb_robustness_outputs'
!mkdir -p "$DRIVE_OUT"

## Install dependencies

If this notebook is inside the repo, upload or clone the repo before running the next cell. If using GitHub, replace the clone URL.

In [ ]:
# Option A: clone your repo after you push it
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git "$PROJECT_DIR"

# Option B: upload the project folder to Colab and set PROJECT_DIR manually
%cd "$PROJECT_DIR"
!python -m pip install --upgrade pip -q
!pip install -r requirements.txt -q

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## EDA figures

In [ ]:
!python -m gtsrb_robustness.explore_data --output-dir outputs/eda

## Train baseline CNN

Use this as the course-method baseline.

In [ ]:
!python -m gtsrb_robustness.train \
  --model baseline_cnn \
  --epochs 12 \
  --batch-size 128 \
  --output-dir runs/baseline_cnn

## Train robust transfer-learning model

This is the main report model: ResNet18 with RandAugment + MixUp.

In [ ]:
!python -m gtsrb_robustness.train \
  --model resnet18 \
  --epochs 25 \
  --batch-size 96 \
  --use-randaugment \
  --mixup-alpha 0.2 \
  --output-dir runs/resnet18_robust

## Evaluate and create report artifacts

In [ ]:
!python -m gtsrb_robustness.evaluate \
  --checkpoint runs/resnet18_robust/best.pt \
  --model resnet18 \
  --output-dir outputs/resnet18_robust

!python -m gtsrb_robustness.plot_history \
  --history runs/resnet18_robust/history.csv \
  --output outputs/resnet18_robust/training_curves.png

!python -m gtsrb_robustness.summarize_results \
  --metrics outputs/resnet18_robust/metrics.json \
  --history runs/resnet18_robust/history.csv \
  --output reports/results_summary.md

## Save outputs to Drive

In [ ]:
!rsync -av runs outputs reports "$DRIVE_OUT"/
print('Saved artifacts to', DRIVE_OUT)